# Gerhardts-style self-consistent Thomas–Fermi solver with top-gate sweep

This upgraded version uses a **single Coulomb kernel** for both the electron Hartree term and the donor/background potential.  
Therefore the positive donor charges are screened by the top gate in the same way as the electron-electron interaction.

It also fixes the movie paths: animations are saved in a local `results/` folder, not in `/mnt/data`.

Main outputs:

- local density, potential, filling factor and compressibility;
- sweep over top-gate distance `dist` from 200 nm to 20000 nm;
- compressibility colormap versus `x` and `dist`;
- two animations: density/potential and local filling factor versus `dist`.

All lengths are in nm, energies in meV, densities in nm^-2.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass, replace
from pathlib import Path
import shutil

from scipy.special import erf
from scipy.optimize import brentq
from scipy.signal import fftconvolve
from matplotlib.animation import FuncAnimation, FFMpegWriter, PillowWriter
from matplotlib.collections import LineCollection
from IPython.display import HTML, display


plt.rcParams.update({"font.size": 13})


## Parameters and units
All lengths are in nm, energies in meV, densities in nm^-2.  
Remember: `1e-3 nm^-2 = 1e11 cm^-2`.

In [ ]:
@dataclass
class Params:
    # geometry
    d: float = 6000.0              # half-width, nm
    Nx: int = 7001                 # use odd number so x=0 is included
    margin: float = 0.999          # avoid exactly x=+-d singularities
    d_S: float = 6050.0      # position on x of surface charge, must be >d

    # material / electrostatics
    eps_r: float = 13.0            # GaAs relative dielectric constant
    n_donor: float = 2.0e-3        # donor density, nm^-2 = 2e11 cm^-2
    n_s: float = 1.e-3        # donor density, nm^-2 = 2e11 cm^-2
    n_target: float = 2.0e-3       # average electron density, nm^-2
    donor_b_fraction: float = 0.999 # donor half-width b = donor_b_fraction*d
    donor_smooth_nm: float = 0.0   # 0 = sharp/uniform; try 20-50 nm to suppress edge artifacts

    # magnetic field / LLs
    B: float = 1.                 # Tesla
    g_spin: float = 0.0            # effective Zeeman as EZ = g_spin * hbar_omega_c; set 0 for spin degeneracy
    n_LL: int = 10
    gamma_rel: float = 0.06        # Gamma/(hbar omega_c) at current B unless use_B_scaling=True
    use_B_scaling: bool = False    # optional: Gamma ~ gamma_rel*hbar_omega_c*sqrt(10T/B)

    # self-consistency
    max_iter: int = 5000
    mix: float = 0.001             # start small; increase only after stable
    tol: float = 2e-8             # density change convergence
    smooth_sigma_nm: float = 0.0   # optional numerical smoothing of Hartree update; 0 disables

    # x-z electrostatic-field map
    field_z_min_nm: float = -600.0
    field_z_max_nm: float = 600.0
    field_Nz: int = 2401
    field_x_padding_nm: float = 1000.0
    field_core_nm: float = 2.      # short-distance regularization of the line-charge kernel
    field_stream_density: float = 2
    field_alpha_low_percentile: float = 1.0   # fields below this percentile are invisible
    field_alpha_high_percentile: float = 99.0  # fields at/above this percentile are fully opaque
    field_alpha_gamma: float = 1.02             # >1 suppresses weak-field lines more strongly

    @property
    def kappa0(self):
        # e^2/(4*pi*eps0*eps_r) in meV nm; 1.44 eV nm = 1440 meV nm
        return 1440.0 / self.eps_r

    @property
    def hbar_omega_c(self):
        # GaAs m*=0.067: hbar omega_c ~= 1.73 meV/T. Original notebook used 1.76.
        return 1.76 * self.B

    @property
    def lB(self):
        # magnetic length in nm
        return 25.658 / np.sqrt(self.B)

    @property
    def Gamma(self):
        if self.use_B_scaling:
            return self.gamma_rel * self.hbar_omega_c * np.sqrt(10.0 / self.B)
        return self.gamma_rel * self.hbar_omega_c

p = Params()

dist_ref = 1e6  # nm



print(p)
print(f"lB = {p.lB:.3f} nm, hbar omega_c = {p.hbar_omega_c:.3f} meV, Gamma = {p.Gamma:.4f} meV")
print(f"Expected average filling = 2*pi*lB^2*n_target = {2*np.pi*p.lB**2*p.n_target:.4f}")
print(f"Same formula: nu = 4.1357*n[1e11 cm^-2]/B[T] = {4.1357*(p.n_target/1e-3)/p.B:.4f}")


## Grid, donor/background potential, and CSG kernel

The CSG/Gerhardts strip kernel is logarithmic and dimensionless.  
The important change in this version is that the donor/background potential is computed numerically from the **same kernel** used for the electron Hartree term:

\[
U_{
m donor}(x_i) =
-

rac{e^2}{\epsilon}\Delta x\sum_j K(x_i,x_j)n_{
m donor}(x_j),
\]

\[
U_H(x_i) =
+

rac{e^2}{\epsilon}\Delta x\sum_j K(x_i,x_j)n_{
m el}(x_j).
\]

Thus the top gate screens both donor-electron and electron-electron electrostatics consistently.


In [ ]:
def make_grid(p: Params):
    x = np.linspace(-p.margin*p.d, p.margin*p.d, p.Nx)
    dx = x[1] - x[0]
    #Nx_S = int(2*p.d_S/dx)
    #x_S = np.linspace(-p.d_S, p.d_S, Nx_S)
    
    x_S = np.arange(-p.d_S, p.d_S, dx)
    
    return x, dx, x_S


def csg_kernel(x, dextra, d, diagonal_cutoff=None):
    """Dimensionless bare CSG logarithmic kernel on [-d,d]."""
    x = np.asarray(x)
    dx_grid = abs(x[1] - x[0]) if diagonal_cutoff is None else diagonal_cutoff
    xi = x[:, None]
    xj = x[None, :]
    d2 = d*d
    ai = np.maximum(d2 - xi*xi, 0.0)
    aj = np.maximum(d2 - xj*xj, 0.0)
    num = np.sqrt(ai*aj) + d2 - xi*xj
    #num = d2
    den = d*np.sqrt( (xi - xj)**2 + dextra**2 )
    den[den == 0.0] = d*dx_grid
    return np.log(np.abs(num/den))

def bare_kernel(x, dextra, diagonal_cutoff=None):
    """Dimensionless bare CSG logarithmic kernel on [-d,d]."""
    x = np.asarray(x)
    dx_grid = abs(x[1] - x[0]) if diagonal_cutoff is None else diagonal_cutoff
    xi = x[:, None]
    xj = x[None, :]
    num = np.sqrt( (xi - xj)**2 + dextra**2 )/dx_grid
    num[num == 0.0] = 2
    return -np.log(np.abs(num))


def kernel_top_gate(x, d, dist, dextra, diagonal_cutoff=None):
    """
    Approximate top-gate screening by image charges.

    The bare y-integrated Coulomb interaction contains -log|x-x'|.
    The image charge subtracts the interaction with a charge displaced by 2*dist in z.

    dist is the metal-2DEG distance in nm. Large dist approaches the bare kernel
    up to an irrelevant additive constant.
    """
    x = np.asarray(x)
    dx_grid = abs(x[1] - x[0]) if diagonal_cutoff is None else diagonal_cutoff
    xi = x[:, None]
    xj = x[None, :]
    delta2 = (xi - xj)**2
    delta2[delta2 == 0.0] = dx_grid**2

    #K0 = csg_kernel(x, dextra, d, diagonal_cutoff=diagonal_cutoff)
    K0 = bare_kernel(x, dextra, diagonal_cutoff=diagonal_cutoff)

    # Image correction in the same logarithmic convention.
    # This reduces the long-range part of the Coulomb interaction.
    K_image_correction = 0.5*np.log(delta2 + (2.0*dist + dextra)**2)
    K = K0 + K_image_correction

    # Additive constants only shift all potentials and are absorbed into mu.
    # This improves numerical conditioning across different dist values.
    #K -= np.min(K)
    return K


def donor_density_profile(x, p: Params):
    """
    Positive donor/background density profile.

    By default this is nearly uniform over the Hall bar, as in the Gerhardts/CSG
    strip model. Set p.donor_smooth_nm ~ 20-50 nm if a sharp donor edge creates
    visible numerical ringing.
    """
    b = p.donor_b_fraction * p.d

    if p.donor_smooth_nm <= 0:
        return p.n_donor * (np.abs(x) <= b).astype(float)

    s = p.donor_smooth_nm
    return 0.5*p.n_donor*(np.tanh((x + b)/s) - np.tanh((x - b)/s))

def s_density_profile(x_S, p: Params):
    b = abs(p.d_S - p.d)/2.
    return p.n_s *p.d_S/((2*np.pi*b**2.)**.5) *(np.exp( - (abs( (x_S-p.d_S)/b)**2.)/2. ) + np.exp( - (abs( (x_S+p.d_S)/b)**2.)/2. ) )


def donor_potential_from_kernel(K, n_donor_x, dx, p: Params):
    """
    Numerical donor/background potential computed from the same kernel as Hartree.

    Positive donor charge attracts electrons, so the electron potential energy is negative.
    """
    return - p.kappa0 * dx * (K @ n_donor_x)


def gate_potential_zero(x):
    return np.zeros_like(x)

def donor_potential_analytic(x, p: Params):
    """Gerhardts background potential: V_bg = -E0 sqrt(1-(x/d)^2)."""
    E0 = 1*np.pi*p.kappa0*2*p.n_target*p.d  # meV; matches 2*pi e^2 n0 d / eps
    return -E0*np.sqrt(np.maximum(1.0 - (x/p.d)**2, 0.0))


def make_kernel_and_background(p: Params, dist=None):
    """
    Build grid, Coulomb kernel, donor density, and donor potential.

    dist=None gives the bare CSG kernel.
    dist=float gives the top-gate screened kernel.
    """
    dist0_d = -150
    x, dx, x_S = make_grid(p)
    K = csg_kernel(x, 0, p.d) if dist is None else kernel_top_gate(x, p.d, dist, 0)
    
    #K_donor = csg_kernel(x, dist0_d, p.d) if dist is None else kernel_top_gate(x, p.d, dist, dist0_d)
    
    K_donor = kernel_top_gate(x, p.d, dist, dist0_d)
    K_donor_down = kernel_top_gate(x, p.d, dist, -dist0_d)
    #K_donor = bare_kernel(x, dist0_d,  diagonal_cutoff=dx)
    #K_donor_down = bare_kernel(x, -dist0_d,  diagonal_cutoff=dx)
    
    K_s = kernel_top_gate(x_S, p.d_S, dist, -0)
    #K_s = bare_kernel(x_S, 0,  diagonal_cutoff=dx)
    #K_s = csg_kernel(x_S, 2*dx, p.d_S)
    nD = donor_density_profile(x, p)
    nS = s_density_profile(x_S, p)
    
    U_donor = donor_potential_from_kernel(K_donor, nD, dx, p)
    U_donor_down = donor_potential_from_kernel(K_donor_down, nD, dx, p)
    
    U_s = donor_potential_from_kernel(K_s, -nS, dx, p)
    indexS_i = np.argmin( np.abs(x_S-x[0]) )
    indexS_f = np.argmin( np.abs(x_S-x[-1]) )
    
    U_s = U_s[indexS_i:indexS_f+1]
    print(indexS_i, indexS_f, len(x), len(x_S), len(U_s), dx)
    
    #U_donor_anal = donor_potential_analytic(x, p)

    U_ext = (U_donor + U_donor_down)*.5 + gate_potential_zero(x) + U_s
    

    # Potential zero is arbitrary; shift for numerical stability.
    U_ext -= np.min(U_ext)
    return x, dx, x_S, K, nD, nS, U_ext

x, dx, x_S, K, nD, nS, U_ext = make_kernel_and_background(p, dist=dist_ref)
UD_anal = donor_potential_analytic(x, p)
fig, ax = plt.subplots(figsize=(7,3))
ax.plot(x*1e-3, U_ext-np.amin(U_ext), label=r"$U_{\rm donor}^{\rm gated}$")
ax.plot(x*1e-3, UD_anal-np.amin(UD_anal), label=r"$U_{\rm donor}^{\rm anal}$", linestyle="dashed", color='orange')
ax.set_ylim(0, 1300)
ax.set_xlabel(r"$x$ [$\mu$m]")
ax.set_ylabel("meV")
ax.grid(True)
ax.legend()
ax2 = ax.twinx()
ax2.plot(x*1e-3, nD/1e-3, color="tab:orange", alpha=0.6, label=r"$n_{\rm donor}$")
ax2.set_ylabel(r"$n_{\rm donor}$ [$10^{11}$ cm$^{-2}$]")
fig.tight_layout()


nD0 = donor_density_profile(x, p)
nS0 = s_density_profile(x, p)
fig1, ax1 = plt.subplots(figsize=(7,3))
ax1.plot(x, nD/1e-3)
ax1.plot(x_S, nS/1e-3)
ax1.set_ylim(0,10)
ax1.grid()




## Landau-level density: the normalization check
For Gaussian-broadened LLs at zero temperature,

\[
n(x)=

rac{1}{2\pi\ell_B^2}\sum_{n,s}


rac12\left[1+\operatorname{erf}\left(

rac{\mu-U(x)-E_{n,s}}{\sqrt2\Gamma}
ight)
ight].
\]

There is no additional `sqrt(2)` prefactor.  A completely filled spin branch contributes exactly `1/(2*pi*lB**2)`.

In [ ]:
def landau_energies(p: Params):
    n = np.arange(p.n_LL, dtype=float)
    E = p.hbar_omega_c*(n + 0.5)
    EZ = p.g_spin*p.hbar_omega_c
    if abs(EZ) < 1e-15:
        # two identical spin branches
        return np.repeat(E, 2)
    return np.sort(np.r_[E - 0.5*EZ, E + 0.5*EZ])


def density_from_potential(U, mu, p: Params):
    E = landau_energies(p)
    arg = (mu - U[:, None] - E[None, :])/(np.sqrt(2.0)*p.Gamma)
    occ = 0.5*(1.0 + erf(arg))
    degeneracy_per_spin = 1.0/(2*np.pi*p.lB**2)
    return degeneracy_per_spin*np.sum(occ, axis=1)


def compressibility_from_potential(U, mu, p: Params):
    """
    Local thermodynamic compressibility in the TFA sense:

        kappa_loc(x) = dn(x)/dmu_loc,   mu_loc = mu - U(x)

    For Gaussian-broadened LLs this is just the local DOS at the Fermi energy.
    Units: nm^-2 / meV.
    """
    E = landau_energies(p)
    z = (mu - U[:, None] - E[None, :])/p.Gamma
    gaussian_dos = np.exp(-0.5*z*z)/(np.sqrt(2*np.pi)*p.Gamma)
    degeneracy_per_spin = 1.0/(2*np.pi*p.lB**2)
    return degeneracy_per_spin*np.sum(gaussian_dos, axis=1)


def filling_from_density(n, p: Params):
    return 2*np.pi*p.lB**2*n

# sanity check: if mu is above exactly m spin-resolved LLs everywhere, nu should be m.
for m in [1,2,4,8]:
    E = landau_energies(p)
    mu_test = E[m-1] + 10*p.Gamma
    n_test = density_from_potential(np.zeros_like(x), mu_test, p)
    print(m, np.mean(filling_from_density(n_test, p)))

## Chemical potential solver
At each self-consistency step, `mu` is adjusted so that the spatially averaged density equals `n_target`.

In [ ]:
def average_density(n, dx, p: Params):
    return dx*np.sum(n)/(2*p.d)


def find_mu_for_target(U, p: Params, dx, verbose=False):
    E = landau_energies(p)
    # Wide bracket: below bottom of sample and above top included LLs.
    lo = np.min(U) + np.min(E) - 20*p.Gamma - 5*p.hbar_omega_c
    hi = np.max(U) + np.max(E) + 20*p.Gamma + 5*p.hbar_omega_c

    def f(mu):
        n = density_from_potential(U, mu, p)
        return average_density(n, dx, p) - p.n_target

    flo, fhi = f(lo), f(hi)
    if verbose:
        print("mu bracket", lo, hi, flo, fhi)
    if flo > 0 or fhi < 0:
        raise RuntimeError(
            f"Cannot bracket mu: f(lo)={flo:g}, f(hi)={fhi:g}. Increase n_LL or check target density."
        )
    return brentq(f, lo, hi, xtol=1e-12, rtol=1e-12, maxiter=100)


## Self-consistent loop
The potential has an arbitrary additive constant.  Here each new potential is shifted so that `min(U)=0`; this is safe because `mu` is recomputed after the shift.

In [ ]:
def gaussian_smooth_realspace(y, sigma_nm, dx):
    if sigma_nm <= 0:
        return y
    # Simple FFT smoothing with open-system caveat; use only as numerical regularization, not physics.
    k = 2*np.pi*np.fft.fftfreq(y.size, d=dx)
    return np.fft.ifft(np.fft.fft(y)*np.exp(-0.5*(sigma_nm*k)**2)).real


def solve_self_consistent(p: Params, dist=None, U_initial=None, verbose=False):
    """
    Self-consistent TFA solver.

    dist=None: bare CSG kernel.
    dist=float: top-gate image-charge screened kernel with gate distance in nm.

    The donor/background potential is computed numerically from the same K used
    in the Hartree term, so donors and electrons are screened consistently by the gate.
    """
    x, dx_local, x_S, K, nD, nS, U_ext = make_kernel_and_background(p, dist=dist)

    U = U_ext.copy() if U_initial is None else U_initial.copy()
    U -= np.min(U)

    mu = find_mu_for_target(U, p, dx_local)
    n = density_from_potential(U, mu, p)

    history = []
    for it in range(p.max_iter):
        U_H = p.kappa0*dx_local*(K @ n)
        U_new = U_ext + U_H
        U_new = gaussian_smooth_realspace(U_new, p.smooth_sigma_nm, dx_local)
        U_new -= np.min(U_new)
        U_mixed = (1-p.mix)*U + p.mix*U_new

        mu_new = find_mu_for_target(U_mixed, p, dx_local)
        n_new = density_from_potential(U_mixed, mu_new, p)

        dn = np.sqrt(dx_local*np.mean((n_new-n)**2))
        if it % 25 == 0 or it == p.max_iter-1:
            history.append((it, mu_new, average_density(n_new, dx_local, p), dn))
            if verbose:
                print(f"it={it:4d}, mu={mu_new:.6g}, n_avg={average_density(n_new, dx_local, p):.6g}, dn={dn:.3e}")

        U, mu, n = U_mixed, mu_new, n_new
        if dn < p.tol:
            history.append((it, mu, average_density(n, dx_local, p), dn))
            break

    comp = compressibility_from_potential(U, mu, p)
    return {
        "x": x,
        "x_S": x_S,
        "dx": dx_local,
        "dist": dist,
        "K": K,
        "n_donor_profile": nD,
        "n_surface_charges": nS,
        "U_ext": U_ext,
        "U": U,
        "mu": mu,
        "n": n,
        "nu": filling_from_density(n, p),
        "compressibility": comp,
        "history": np.array(history),
    }


# Single reference run. Change dist_ref to test one gate distance.
res = solve_self_consistent(p, dist=dist_ref)
print("dist_ref:", dist_ref, "nm")
print("iterations logged:", res["history"][-5:])
print("average density:", average_density(res["n"], res["dx"], p), "target:", p.n_target)
print("average donor density:", average_density(res["n_donor_profile"], res["dx"], p))
print("center filling:", res["nu"][len(res["nu"])//2])

## Diagnostics
These plots are the first things to inspect when the density looks wrong.

In [ ]:
def plot_solution(res, p: Params):
    x = res["x"]
    U = res["U"]
    mu = res["mu"]
    n = res["n"]
    nu = res["nu"]
    hwc = p.hbar_omega_c

    fig, ax = plt.subplots(figsize=(11,3.6))
    ax.plot(x*1e-3, (U-np.amin(U))/hwc, color="tab:red", lw=2, label=r"$U/\hbar\omega_c$")
    ax.axhline(mu/hwc, color="tab:green", lw=1.5, label=r"$\mu/\hbar\omega_c$")
    ax.set_xlabel(r"$x$ [$\mu$m]")
    ax.set_ylabel(r"energy / $\hbar\omega_c$")
    ax.grid(True)
    ax.set_ylim(0,10)
    ax2 = ax.twinx()
    ax2.plot(x*1e-3, n/1e-3, color="tab:blue", lw=2, label=r"$n$ [$10^{11}$ cm$^{-2}$]")
    ax2.set_ylabel(r"$n$ [$10^{11}$ cm$^{-2}$]", color="tab:blue")
    ax.legend(loc="upper left")
    fig.tight_layout()

    plt.figure(figsize=(11,3.2))
    plt.plot(x*1e-3, nu, lw=2)
    for m in range(0, int(np.nanmax(nu))+2):
        plt.axhline(m, color="0.7", lw=0.8)
    plt.xlabel(r"$x$ [$\mu$m]")
    plt.ylabel(r"$\nu(x)=2\pi\ell_B^2 n(x)$")
    plt.grid(True)
    plt.tight_layout()
    
    plt.figure(figsize=(11,3.2))
    plt.plot(x*1e-3, nu, lw=2)
    for m in range(0, int(np.nanmax(nu))+2):
        plt.axhline(m, color="0.7", lw=0.8)
    plt.xlabel(r"$x$ [$\mu$m]")
    plt.ylabel(r"$\nu(x)=2\pi\ell_B^2 n(x)$")
    plt.xlim(-p.d*1e-3, -p.d*1e-3*.7)
    plt.grid(True)
    plt.tight_layout()

    """plt.figure(figsize=(11,3.2))
    plt.plot(x*1e-3, (mu-U)/hwc, lw=2, label=r"$(\mu-U)/\hbar\omega_c$")
    for m in range(p.n_LL):
        plt.axhline(m+0.5, color="0.75", lw=0.8)
    plt.xlabel(r"$x$ [$\mu$m]")
    plt.ylabel("local Fermi energy / LL scale")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()"""

plot_solution(res, p)

## Full electrostatic potential and electric field in the x-z plane

The original solver only needs the potential energy at the 2DEG plane, z=0.  Here the same y-integrated Coulomb model is extended to an arbitrary observation point (x,z).

For a source line at (x',z_s) and a grounded top gate at z=z_g, the dimensionless Green function is

\[
K(x,z;x',z_s)=\frac12\log\!\left[
\frac{(x-x')^2+(z-2z_g+z_s)^2}
     {(x-x')^2+(z-z_s)^2}
\right],
\]

with a short-distance numerical regularization controlled by `field_core_nm`.  This is the same direct-charge plus opposite-sign image-charge construction used by `kernel_top_gate`.

The signed number-density layers are:

- electrons: `-n_e(x)` at z=0;
- upper and lower donors: `+n_D(x)/2` at z=+90 nm and z=-90 nm;
- surface charge: `-n_S(x)` at z=0 near x=+-d_S.

The electrostatic potential is returned in mV (`1 meV/e = 1 mV`). Consequently, `-gradient(phi)` in mV/nm is numerically equal to the electric field in MV/m.

### Field-only visualization

The final figure deliberately does **not** display the potential. Electric-field streamlines are drawn on a white background. Their local opacity is determined by the normalized field modulus $|\mathbf E(x,z)|$: weak-field portions become transparent, while strong-field portions remain visible. The low/high percentile cutoffs and nonlinear contrast are controlled by `field_alpha_low_percentile`, `field_alpha_high_percentile`, and `field_alpha_gamma`.


In [ ]:
def double_grounded_gate_electric_kernel_1d(
    delta_x,
    z_obs,
    z_src,
    dist,
    core_nm,
    n_images=50,
):
    """
    Analytic electric-field Green function for two grounded parallel
    metallic plates located at

        z = +dist
        z = -dist.

    A source at z_src generates the infinite image-charge families

        same-sign:
            z_same(n) = z_src + 4*n*dist

        opposite-sign:
            z_opp(n)  = -z_src + (4*n + 2)*dist

    with n = ..., -2, -1, 0, 1, 2, ...

    The physical electric field is obtained directly from -grad(phi).

    Parameters
    ----------
    delta_x : array
        x_obs - x_src, in nm.

    z_obs : float
        Observation height, in nm.

    z_src : float
        Source-layer height, in nm.

    dist : float
        Half-distance between the two plates.
        Plates are at z = +/- dist.

    core_nm : float
        Short-distance regularization.

    n_images : int
        Number of positive/negative image orders retained.
        The sum runs from -n_images to +n_images.

    Returns
    -------
    K_Ex, K_Ez : arrays
        Electric-field Green-function components, in 1/nm.
    """

    K_Ex = np.zeros_like(delta_x, dtype=float)
    K_Ez = np.zeros_like(delta_x, dtype=float)

    for n in range(-n_images, n_images + 1):

        # --------------------------------------------------
        # Same-sign image family
        # Includes the real charge for n = 0
        # --------------------------------------------------
        z_same = z_src + 4.0 * n * dist

        dz_same = z_obs - z_same

        r2_same = (
            delta_x**2
            + dz_same**2
            + core_nm**2
        )

        K_Ex += delta_x / r2_same
        K_Ez += dz_same / r2_same


        # --------------------------------------------------
        # Opposite-sign image family
        # --------------------------------------------------
        z_opp = -z_src + (4.0 * n + 2.0) * dist

        dz_opp = z_obs - z_opp

        r2_opp = (
            delta_x**2
            + dz_opp**2
            + core_nm**2
        )

        # Opposite charge -> subtract its field
        K_Ex -= delta_x / r2_opp
        K_Ez -= dz_opp / r2_opp


    return K_Ex, K_Ez

In [ ]:
def grounded_gate_kernel_1d(delta_x, z_obs, z_src, z_gate, core_nm):
    """Translation-invariant y-integrated Green function for a grounded plane at z_gate."""
    direct_r2 = delta_x**2 + (z_obs - z_src)**2 + core_nm**2
    z_image = 2.0*z_gate - z_src
    image_r2 = delta_x**2 + (z_obs - z_image)**2 + core_nm**2
    return 0.5*np.log(image_r2/direct_r2)

    
def grounded_gate_electric_kernel_1d(
    delta_x,
    z_obs,
    z_src,
    z_gate,
    core_nm,
):
    """
    Analytic electric-field Green function for a grounded metallic plane
    located at z = z_gate.
    A real source at z_src has an opposite-sign image at

        z_img = 2*z_gate - z_src.
    The corresponding potential Green function is

        K = 0.5 * log(r_img^2 / r_dir^2),
    with
        r_dir^2 = dx^2 + (z_obs-z_src)^2 + core_nm^2
        r_img^2 = dx^2 + (z_obs-z_img)^2 + core_nm^2.
    The physical electric field is E = -grad(K).
    """
    # Position of the image charge
    z_img = 2.0 * z_gate - z_src
    # Vertical separations
    dz_dir = z_obs - z_src
    dz_img = z_obs - z_img
    # Regularized squared distances
    r2_dir = (delta_x**2 + dz_dir**2 + core_nm**2)
    r2_img = (delta_x**2 + dz_img**2 + core_nm**2)
    # E = -grad(K)
    K_Ex = delta_x / r2_dir - delta_x / r2_img
    K_Ez = dz_dir / r2_dir - dz_img / r2_img
    return K_Ex, K_Ez


def _odd_uniform_grid(x_min, x_max, dx):
    n = int(np.ceil((x_max - x_min)/dx)) + 1
    if n % 2 == 0:
        n += 1
    center = 0.5*(x_min + x_max)
    return center + (np.arange(n) - n//2)*dx


def compute_xz_electrostatics(res, p: Params, z_min_nm=None, z_max_nm=None, Nz=None):
    """
    Compute the electrostatic potential phi(x,z) and E=-grad(phi).

    phi is in mV and Ex, Ez are in MV/m. The potential is computed with the
    original Green-function convolution, while Ex and Ez are computed directly
    from the analytic electric-field Green function (-grad K), without taking
    a numerical gradient of phi.
    """
    x = np.asarray(res["x"])
    x_S = np.asarray(res["x_S"])
    dx = float(res["dx"])

    z_min_nm = p.field_z_min_nm if z_min_nm is None else float(z_min_nm)
    z_max_nm = p.field_z_max_nm if z_max_nm is None else float(z_max_nm)
    Nz = p.field_Nz if Nz is None else int(Nz)
    z = np.linspace(z_min_nm, z_max_nm, Nz)

    x_extent = max(np.max(np.abs(x)), np.max(np.abs(x_S))) + p.field_x_padding_nm
    x_field = _odd_uniform_grid(-x_extent, x_extent, dx)

    # Put all layers on one common uniform grid. Values outside their native domains are zero.
    n_e = np.interp(x_field, x, res["n"], left=0.0, right=0.0)
    n_D = np.interp(x_field, x, res["n_donor_profile"], left=0.0, right=0.0)
    n_S = np.interp(x_field, x_S, res["n_surface_charges"], left=0.0, right=0.0)

    # The old kernel argument dextra corresponds to a physical source position z_src=-dextra.
    donor_offset_nm = 150.0
    z_gate = float(res["dist"]) if res["dist"] is not None else 1.0e12
    if z_gate <= z_max_nm:
        raise ValueError(
            f"The plotting domain reaches z={z_max_nm:g} nm, but the grounded gate is at "
            f"z={z_gate:g} nm. Set field_z_max_nm below the gate."
        )

    # Signed charge-number densities in units of e per nm^2. =============================                          # DENSITY define the layer set with the charge density that you want
    layers = [                                                                          
        (-n_e, 0.0, "electrons"),
        (+0.5*n_D, +donor_offset_nm, "upper donors"),
        (+0.5*n_D, -donor_offset_nm, "lower donors"),
        #(-n_S, 0.0, "surface charge"),
    ]
    layers = [
        (-n_e, 0.0, "electrons"),
        (+1.0*n_D, +donor_offset_nm, "upper donors"),
        #(+0.5*n_D, -donor_offset_nm, "lower donors"),
    ]

    delta_x = (np.arange(x_field.size) - x_field.size//2)*dx
    phi_mV = np.zeros((z.size, x_field.size), dtype=float)
    Ex_MVm = np.zeros_like(phi_mV)
    Ez_MVm = np.zeros_like(phi_mV)

    # Compute potential and electric field from the same charge layers.
    # phi uses the original potential Green function; Ex and Ez are computed
    # directly by convolution with its analytic derivative (-grad K), avoiding
    # any numerical differentiation of phi.
    for iz, z_obs in enumerate(z):
        phi_row = np.zeros_like(x_field)
        Ex_row = np.zeros_like(x_field)
        Ez_row = np.zeros_like(x_field)

        for rho_signed, z_src, _ in layers:
            kernel_phi = grounded_gate_kernel_1d(
                delta_x, z_obs=z_obs, z_src=z_src,
                z_gate=z_gate, core_nm=p.field_core_nm,
            )
            #=====================================                                                                      #FIELD Electric field kernel computed for 1 or 2 metallic plates
            
            kernel_Ex, kernel_Ez = grounded_gate_electric_kernel_1d(delta_x, z_obs=z_obs, z_src=z_src,z_gate=z_gate, core_nm=p.field_core_nm,)
            #kernel_Ex, kernel_Ez = double_grounded_gate_electric_kernel_1d(delta_x, z_obs=z_obs, z_src=z_src,dist=z_gate, core_nm=p.field_core_nm,)
            

            phi_row += fftconvolve(rho_signed, kernel_phi, mode="same")*dx
            Ex_row += fftconvolve(rho_signed, kernel_Ex, mode="same")*dx
            Ez_row += fftconvolve(rho_signed, kernel_Ez, mode="same")*dx

        phi_mV[iz] = p.kappa0*phi_row
        Ex_MVm[iz] = -p.kappa0*Ex_row
        Ez_MVm[iz] = -p.kappa0*Ez_row

    # Remove one arbitrary global potential constant.  This has no effect on
    # the analytically computed electric field.
    phi_mV -= np.nanmean(phi_mV)

    Eabs_MVm = np.hypot(Ex_MVm, Ez_MVm)

    return {
        "x_nm": x_field,
        "z_nm": z,
        "phi_mV": phi_mV,
        "Ex_MVm": Ex_MVm,
        "Ez_MVm": Ez_MVm,
        "Eabs_MVm": Eabs_MVm,
        "n_e": n_e,
        "n_D": n_D,
        "n_S": n_S,
        "donor_offset_nm": donor_offset_nm,
        "z_gate_nm": z_gate,
    }


def _add_density_line(ax, x_um, z_um, density, color, linewidth, label, alpha_floor=0.005):
    """Draw a thick horizontal charge layer with local opacity set by its density."""
    density = np.asarray(density)
    scale = np.nanmax(np.abs(density))
    if not np.isfinite(scale) or scale <= 0:
        return
    alpha = alpha_floor + (1.0-alpha_floor)*np.clip(np.abs(density)/scale, 0.0, 1.0)
    points = np.column_stack([x_um, np.full_like(x_um, z_um)]).reshape(-1, 1, 2)
    segments = np.concatenate([points[:-1], points[1:]], axis=1)
    rgba = np.tile(np.array(plt.matplotlib.colors.to_rgba(color)), (segments.shape[0], 1))
    rgba[:, 3] = 0.5*(alpha[:-1] + alpha[1:])
    collection = LineCollection(segments, colors=rgba, linewidths=linewidth, zorder=8)
    ax.add_collection(collection)
    ax.plot([], [], color=color, lw=linewidth, label=label)


def plot_xz_field(field, p: Params, xlim_um=None, zlim_um=None):
    """Electric-field streamlines only, with local opacity proportional to |E|."""
    x_um = field["x_nm"]*1e-3
    z_um = field["z_nm"]*1e-3

    Ex = field["Ex_MVm"]
    Ez = field["Ez_MVm"]
    Emag = field["Eabs_MVm"]

    # Robust normalization of |E|. Values below the lower percentile vanish;
    # values above the upper percentile are fully opaque.
    finite_positive = Emag[np.isfinite(Emag) & (Emag > 0)]
    if finite_positive.size == 0:
        raise ValueError("The computed electric field contains no finite positive values.")

    E_low = np.nanpercentile(finite_positive, p.field_alpha_low_percentile)
    E_high = np.nanpercentile(finite_positive, p.field_alpha_high_percentile)
    if not np.isfinite(E_high) or E_high <= E_low:
        E_low = 0.0
        E_high = np.nanmax(finite_positive)

    visibility = np.clip((Emag - E_low)/(E_high - E_low), 0.0, 1.0)
    visibility = visibility**p.field_alpha_gamma

    # A black colormap whose alpha channel runs from 0 to 1. Since streamplot
    # samples the 2D `visibility` array along each trajectory, both line segments
    # and arrowheads fade away in weak-field regions.
    rgba = np.zeros((256, 4), dtype=float)
    rgba[:, :3] = 0.0
    rgba[:, 3] = np.linspace(0.0, 1.0, 256)
    alpha_cmap = plt.matplotlib.colors.ListedColormap(rgba, name="field_alpha_black")
    alpha_norm = plt.matplotlib.colors.Normalize(vmin=0.0, vmax=1.0)

    fig, ax = plt.subplots(figsize=(13, 5.5))
    ax.set_facecolor("white")

    # Draw the streamline curves, but suppress streamplot's automatically
    # positioned arrowheads. Arrow directions are added below on a regular grid.
    """ax.streamplot(
        x_um, z_um, Ex, Ez,
        density=p.field_stream_density,
        color=visibility,
        cmap=alpha_cmap,
        norm=alpha_norm,
        linewidth=1.,   #size automatic lines of matplot
        arrowsize=0.5, #size arrowhead of automatic matplot
        minlength=0.08,
        zorder=5,
    )"""

    # ------------------------------------------------------------------
    # Regularly positioned field-direction arrows
    # ------------------------------------------------------------------
    # Use evenly spaced sample positions in x and z, independently of the
    # underlying numerical grid resolution. This keeps arrowheads visually
    # aligned instead of allowing streamplot to place them at irregular
    # positions along different trajectories.
    arrow_Nx = 50
    arrow_Nz = 20

    ix = np.unique(np.linspace(0, x_um.size - 1, arrow_Nx).astype(int))
    iz = np.unique(np.linspace(0, z_um.size - 1, arrow_Nz).astype(int))

    Xq, Zq = np.meshgrid(x_um[ix], z_um[iz])
    Exq = Ex[np.ix_(iz, ix)]
    Ezq = Ez[np.ix_(iz, ix)]
    Eq = Emag[np.ix_(iz, ix)]
    Vq = visibility[np.ix_(iz, ix)]

    # Normalize the vectors so arrow length encodes direction only; the local
    # field strength is retained through the opacity, exactly as for the lines.
    denom = np.where(Eq > 1e-6, Eq, 1)
    Uq = Exq / (denom+1e-10)
    Wq = Ezq / (denom+1e-10)

    # Black arrows with local alpha inherited from the same |E|-visibility map.
    arrow_rgba = np.zeros(Vq.shape + (4,), dtype=float)
    arrow_rgba[..., 3] = np.clip(Vq, 0.0, 1.0)

    ax.quiver(
        Xq, Zq, Uq, Wq,
        color=arrow_rgba.reshape(-1, 4),
        angles="uv",
        scale_units="width",
        scale=60,
        width=0.003,
        headwidth=2,
        headlength=3.0,
        headaxislength=2.5,
        pivot="middle",
        zorder=6,
    )

    zD_um = field["donor_offset_nm"]*1e-3
    _add_density_line(ax, x_um, 0.0, field["n_e"], "tab:blue", 9.0, "electrons")                                   #   #THICK LINES add the thick lines representing the used charge density
    #_add_density_line(ax, x_um, 0.0, field["n_D"], "tab:blue", 9.0, "electrons")
    _add_density_line(ax, x_um, +zD_um, 0.5*field["n_D"], "tab:red", 8.0, "upper donors")
    #_add_density_line(ax, x_um, -zD_um, 0.5*field["n_D"], "tab:red", 8.0, "lower donors")
    #_add_density_line(ax, x_um, 0.0, field["n_S"], "purple", 11.0, "surface charge")

    if np.isfinite(field["z_gate_nm"]) and field["z_gate_nm"] < 1e10:
        zg_um = field["z_gate_nm"]*1e-3
        if z_um.min() <= zg_um <= z_um.max():
            ax.axhline(zg_um, color="black", lw=3, label="grounded top gate")


    z_gate = res["dist"]*1e-3
    ax.axhline(
        z_gate,
        color="gold",
        linewidth=8.0,
        solid_capstyle="butt",
        zorder=20,
        label="Metallic top gate",
    )
    #ax.axhline(-z_gate,color="gold",linewidth=8.0,solid_capstyle="butt",zorder=20,label="Metallic top gate",)                    # LOWER PLATE lower metallic plate, if present


    ax.set_xlabel(r"$x$ [$\mu$m]")
    ax.set_ylabel(r"$z$ [$\mu$m]")
    ax.set_title(r"Electric-field lines; opacity $\propto |\mathbf{E}|$")
    ax.set_aspect("auto")
    ax.grid(False)
    #ax.legend(loc="upper right", framealpha=0.92, ncol=2)

    if xlim_um is not None:
        ax.set_xlim(*xlim_um)
    if zlim_um is not None:
        ax.set_ylim(*zlim_um)
    else:
        ax.set_ylim(z_um.min(), z_um.max())
        
    #ax.set_ylim(z_um.min(), z_gate)
    ax.set_ylim(-.6, .6)

    # Store normalization values on the axes for reproducibility/debugging.
    ax._field_visibility_limits_MVm = (E_low, E_high)
    fig.tight_layout()
    return fig, ax


field_xz = compute_xz_electrostatics(res, p)
print(
    "Field map shape:", field_xz["phi_mV"].shape,
    "| max |E| =", np.nanmax(field_xz["Eabs_MVm"]), "MV/m"
)
fig_field, ax_field = plot_xz_field(field_xz, p)
print(
    "Opacity normalization: |E| =",
    ax_field._field_visibility_limits_MVm[0], "to",
    ax_field._field_visibility_limits_MVm[1], "MV/m"
)


plt.savefig("field_lines.pdf")
plt.show()


In [ ]:
# ============================================================
# Induced charge density on the TOP metallic plate, z = +dist
# ============================================================

# ------------------------------------------------------------
# Constants
# ------------------------------------------------------------

eps0 = 8.8541878128e-12       # F/m
e_charge = 1.602176634e-19    # C

# Relative dielectric constant of the material between the
# charges and the metallic plate.
#
# Use the same dielectric constant as in the electrostatics.
# For GaAs, for example, ~12.4-12.9 depending on convention.
epsilon_r = 12.4


# ------------------------------------------------------------
# Field data
# ------------------------------------------------------------

x_nm = np.asarray(field_xz["x_nm"])
z_nm = np.asarray(field_xz["z_nm"])

Ez_MVm = np.asarray(field_xz["Ez_MVm"])

dist_nm = float(res["dist"])


# ------------------------------------------------------------
# Find the grid point immediately BELOW the upper metallic plate
# ------------------------------------------------------------

below = np.where(z_nm < dist_nm)[0]

if len(below) == 0:
    raise ValueError(
        "The z grid contains no point below the metallic plate."
    )

iz = below[-1]

z_eval_nm = z_nm[iz]

print(f"Metal plate: z = {dist_nm:.6f} nm")
print(f"Field evaluated at z = {z_eval_nm:.6f} nm")
print(f"distance from plate = {dist_nm-z_eval_nm:.6f} nm")


# ------------------------------------------------------------
# Normal electric field
#
# Ez is stored in MV/m -> convert to V/m
# ------------------------------------------------------------

Ez_top_Vm = Ez_MVm[iz, :] * 1e6


# ------------------------------------------------------------
# Surface charge
#
# For the LOWER face of the TOP conductor:
#
#     n_out = -z_hat
#
# therefore
#
#     sigma = epsilon E . n_out
#           = -epsilon Ez
# ------------------------------------------------------------

sigma_Cm2 = -eps0 * epsilon_r * Ez_top_Vm


# ------------------------------------------------------------
# Equivalent electron-number surface density
# ------------------------------------------------------------

# signed number of elementary charges per m^2
sigma_over_e_m2 = sigma_Cm2 / e_charge

# convert m^-2 -> cm^-2
sigma_over_e_cm2 = sigma_over_e_m2 * 1e-4

# convert m^-2 -> nm^-2
sigma_over_e_nm2 = sigma_over_e_m2 * 1e-18


# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------

x_um = x_nm * 1e-3

fig, ax = plt.subplots(figsize=(11, 4.5))

ax.plot(
    x_um,
    sigma_over_e_cm2*1e-11,
    linewidth=2.0,
)

ax.axhline(
    0.0,
    color="black",
    linewidth=0.8,
    alpha=0.5,
)

ax.set_xlabel(r"$x$ [$\mu$m]")
ax.set_ylabel(
    r"$\sigma_{\rm top}/e$ [$10^{11}$ cm$^{-2}$]"
)

ax.set_title(
    rf"Induced surface charge on metallic plate at "
    rf"$z={dist_nm*1e-3:.3f}\,\mu$m"
)

ax.grid(alpha=0.2)

ax.set_ylim(-1.5,1)
ax.set_xlim(x_um.min(), x_um.max())

plt.tight_layout()
plt.savefig("induced_charge.pdf")
plt.show()


# ------------------------------------------------------------
# Some diagnostics
# ------------------------------------------------------------

print()
print("Surface-charge statistics:")
print(
    "min sigma/e =",
    f"{np.min(sigma_over_e_cm2):.4e}",
    "cm^-2"
)
print(
    "max sigma/e =",
    f"{np.max(sigma_over_e_cm2):.4e}",
    "cm^-2"
)

# Translationally invariant in y:
# integral over x gives charge per unit length in y.
Q_per_length_Cm = np.trapz(
    sigma_Cm2,
    x_nm * 1e-9,
)

N_per_length_m = Q_per_length_Cm / e_charge

print()
print(
    "Integrated induced charge per unit y-length:",
    f"{Q_per_length_Cm:.4e}",
    "C/m"
)
print(
    "Equivalent elementary charges per unit y-length:",
    f"{N_per_length_m:.4e}",
    "1/m"
)

In [ ]:
raise SystemExit("Stopping notebook here.")

In [ ]:
def plot_xz_field(field, p: Params, xlim_um=None, zlim_um=None):
    """Electric-field arrows with opacity and length proportional to |E|."""

    x_um = field["x_nm"]*1e-3
    z_um = field["z_nm"]*1e-3

    Ex = field["Ex_MVm"]
    Ez = field["Ez_MVm"]
    Emag = field["Eabs_MVm"]

    # ---------------------------------------------------------
    # Visibility / opacity normalization
    # ---------------------------------------------------------

    finite_positive = Emag[np.isfinite(Emag) & (Emag > 0)]

    if finite_positive.size == 0:
        raise ValueError(
            "The computed electric field contains no finite positive values."
        )

    E_low = np.nanpercentile(
        finite_positive,
        p.field_alpha_low_percentile
    )

    E_high = np.nanpercentile(
        finite_positive,
        p.field_alpha_high_percentile
    )

    if not np.isfinite(E_high) or E_high <= E_low:
        E_low = 0.0
        E_high = np.nanmax(finite_positive)

    visibility = np.clip(
        (Emag - E_low) / (E_high - E_low),
        0.0,
        1.0
    )

    visibility = visibility**p.field_alpha_gamma

    # ---------------------------------------------------------
    # Alpha colormap
    # ---------------------------------------------------------

    rgba = np.zeros((256, 4), dtype=float)
    rgba[:, :3] = 0.0
    rgba[:, 3] = np.linspace(0.0, 1.0, 256)

    alpha_cmap = plt.matplotlib.colors.ListedColormap(
        rgba,
        name="field_alpha_black"
    )

    alpha_norm = plt.matplotlib.colors.Normalize(
        vmin=0.0,
        vmax=1.0
    )

    fig, ax = plt.subplots(figsize=(13, 5.5))
    ax.set_facecolor("white")


    # ---------------------------------------------------------
    # Regular arrow grid
    # ---------------------------------------------------------

    arrow_Nx = 100
    arrow_Nz = 80

    ix = np.unique(
        np.linspace(
            0,
            x_um.size - 1,
            arrow_Nx
        ).astype(int)
    )

    iz = np.unique(
        np.linspace(
            0,
            z_um.size - 1,
            arrow_Nz
        ).astype(int)
    )

    Xq, Zq = np.meshgrid(
        x_um[ix],
        z_um[iz]
    )

    Exq = Ex[np.ix_(iz, ix)]
    Ezq = Ez[np.ix_(iz, ix)]

    Eq = Emag[np.ix_(iz, ix)]
    Vq = visibility[np.ix_(iz, ix)]


    # ---------------------------------------------------------
    # Unit-vector field direction
    # ---------------------------------------------------------

    denom = np.maximum(Eq, 1e-30)

    Udir = Exq / denom
    Wdir = Ezq / denom


    # ---------------------------------------------------------
    # Arrow LENGTH proportional to |E|
    #
    # Robust percentile normalization prevents a few extremely
    # strong points from making every other arrow microscopic.
    # ---------------------------------------------------------

    Eq_positive = Eq[
        np.isfinite(Eq) & (Eq > 0)
    ]

    E_arrow_low = np.nanpercentile(
        Eq_positive,
        10.0
    )

    E_arrow_high = np.nanpercentile(
        Eq_positive,
        98.0
    )

    if (
        not np.isfinite(E_arrow_high)
        or E_arrow_high <= E_arrow_low
    ):
        E_arrow_low = 0.0
        E_arrow_high = np.nanmax(Eq_positive)

    Lq = np.clip(
        (Eq - E_arrow_low)
        /
        (E_arrow_high - E_arrow_low),
        0.0,
        1.0
    )

    # Sublinear mapping:
    # < 1 makes moderate fields more visible
    arrow_length_gamma = 0.5
    Lq = Lq**arrow_length_gamma

    # Optional minimum length for arrows that are already visible.
    arrow_min_length = 0.05

    Lq = np.where(
        Vq > 0.02,
        arrow_min_length
        + (1.0 - arrow_min_length)*Lq,
        0.0
    )

    Uq = Udir * Lq
    Wq = Wdir * Lq


    # ---------------------------------------------------------
    # Arrow opacity
    # ---------------------------------------------------------

    arrow_rgba = np.zeros(
        Vq.shape + (4,),
        dtype=float
    )

    arrow_rgba[..., 3] = np.clip(
        Vq,
        0.0,
        1.0
    )


    # ---------------------------------------------------------
    # Quiver
    # ---------------------------------------------------------

    ax.quiver(
        Xq,
        Zq,
        Uq,
        Wq,
        color=arrow_rgba.reshape(-1, 4),
        angles="xy",
        scale_units="xy",
        scale=20.0,
        width=0.0025,
        headwidth=2,
        headlength=4.0,
        headaxislength=4.5,
        pivot="middle",
        zorder=6,
    )


    # ---------------------------------------------------------
    # Charge distributions
    # ---------------------------------------------------------

    zD_um = field["donor_offset_nm"]*1e-3

    _add_density_line(
        ax,
        x_um,
        0.0,
        field["n_e"],
        "tab:blue",
        9.0,
        "electrons"
    )

    _add_density_line(
        ax,
        x_um,
        +zD_um,
        0.5*field["n_D"],
        "tab:red",
        8.0,
        "upper donors"
    )

    _add_density_line(
        ax,
        x_um,
        -zD_um,
        0.5*field["n_D"],
        "tab:red",
        8.0,
        "lower donors"
    )

    _add_density_line(
        ax,
        x_um,
        0.0,
        field["n_S"],
        "purple",
        11.0,
        "surface charge"
    )


    # ---------------------------------------------------------
    # Metallic gate
    # ---------------------------------------------------------

    if (
        np.isfinite(field["z_gate_nm"])
        and field["z_gate_nm"] < 1e10
    ):

        zg_um = field["z_gate_nm"]*1e-3

        if z_um.min() <= zg_um <= z_um.max():

            ax.axhline(
                zg_um,
                color="black",
                lw=3,
                label="grounded top gate"
            )


    z_gate = res["dist"]*1e-3

    ax.axhline(
        z_gate,
        color="gold",
        linewidth=8.0,
        solid_capstyle="butt",
        zorder=20,
        label="Metallic top gate",
    )


    # ---------------------------------------------------------
    # Axes
    # ---------------------------------------------------------

    ax.set_xlabel(r"$x$ [$\mu$m]")
    ax.set_ylabel(r"$z$ [$\mu$m]")

    ax.set_title(
        r"Electric field: arrow opacity and length "
        r"$\propto |\mathbf{E}|$"
    )

    ax.set_aspect("auto")
    ax.grid(False)

    if xlim_um is not None:
        ax.set_xlim(*xlim_um)

    if zlim_um is not None:
        ax.set_ylim(*zlim_um)
    else:
        ax.set_ylim(z_um.min(), z_um.max())

    ax.set_ylim(-.6, .6)

    ax._field_visibility_limits_MVm = (
        E_low,
        E_high
    )

    fig.tight_layout()

    return fig, ax


fig_field, ax_field = plot_xz_field(field_xz, p)
plt.savefig("field_lines.pdf")
plt.show()

In [ ]:
def plot_xz_field_lines(field, p: Params, xlim_um=None, zlim_um=None):
    """
    Plot continuous electric-field lines without arrowheads.

    Line opacity is proportional to |E|, as in the previous visualization.
    """

    x_um = field["x_nm"] * 1e-3
    z_um = field["z_nm"] * 1e-3

    Ex = field["Ex_MVm"]
    Ez = field["Ez_MVm"]
    Emag = field["Eabs_MVm"]

    # ---------------------------------------------------------
    # Robust normalization of |E| for opacity
    # ---------------------------------------------------------

    finite_positive = Emag[
        np.isfinite(Emag) & (Emag > 0)
    ]

    if finite_positive.size == 0:
        raise ValueError(
            "The computed electric field contains no finite positive values."
        )

    E_low = np.nanpercentile(
        finite_positive,
        p.field_alpha_low_percentile
    )

    E_high = np.nanpercentile(
        finite_positive,
        p.field_alpha_high_percentile
    )

    if not np.isfinite(E_high) or E_high <= E_low:
        E_low = 0.0
        E_high = np.nanmax(finite_positive)

    visibility = np.clip(
        (Emag - E_low) / (E_high - E_low),
        0.0,
        1.0
    )

    visibility = visibility**p.field_alpha_gamma


    # ---------------------------------------------------------
    # Black colormap with variable alpha
    # ---------------------------------------------------------

    rgba = np.zeros((256, 4), dtype=float)

    # black RGB
    rgba[:, :3] = 0.0

    # alpha 0 -> 1
    rgba[:, 3] = np.linspace(
        0.0,
        1.0,
        256
    )

    alpha_cmap = plt.matplotlib.colors.ListedColormap(
        rgba,
        name="field_alpha_black"
    )

    alpha_norm = plt.matplotlib.colors.Normalize(
        vmin=0.0,
        vmax=1.0
    )


    # ---------------------------------------------------------
    # Figure
    # ---------------------------------------------------------

    fig, ax = plt.subplots(
        figsize=(13, 5.5)
    )

    ax.set_facecolor("white")


    # ---------------------------------------------------------
    # Continuous field lines
    #
    # arrowsize is made essentially zero so only the
    # continuous trajectories remain visible.
    # ---------------------------------------------------------

    stream = ax.streamplot(
        x_um,
        z_um,
        Ex,
        Ez,

        density=p.field_stream_density,

        color=visibility,
        cmap=alpha_cmap,
        norm=alpha_norm,

        linewidth=2,

        # effectively suppress arrowheads
        arrowsize=0.000,

        minlength=0.02,

        # useful to avoid extremely long winding trajectories
        maxlength=20.0,

        integration_direction="both",

        broken_streamlines=True,

        zorder=5,
    )


    # ---------------------------------------------------------
    # Charge distributions
    # ---------------------------------------------------------

    zD_um = field["donor_offset_nm"] * 1e-3

    _add_density_line(
        ax,
        x_um,
        0.0,
        field["n_e"],
        "tab:blue",
        9.0,
        "electrons"
    )

    _add_density_line(
        ax,
        x_um,
        +zD_um,
        0.5 * field["n_D"],
        "tab:red",
        8.0,
        "upper donors"
    )

    _add_density_line(
        ax,
        x_um,
        -zD_um,
        0.5 * field["n_D"],
        "tab:red",
        8.0,
        "lower donors"
    )

    _add_density_line(
        ax,
        x_um,
        0.0,
        field["n_S"],
        "purple",
        11.0,
        "surface charge"
    )


    # ---------------------------------------------------------
    # Metallic plate(s)
    # ---------------------------------------------------------

    z_gate = res["dist"] * 1e-3

    ax.axhline(
        z_gate,
        color="gold",
        linewidth=8.0,
        solid_capstyle="butt",
        zorder=20,
        label="Metallic top gate",
    )

    # For double metallic plates:
    #
    # ax.axhline(
    #     -z_gate,
    #     color="gold",
    #     linewidth=8.0,
    #     solid_capstyle="butt",
    #     zorder=20,
    # )


    # ---------------------------------------------------------
    # Axes
    # ---------------------------------------------------------

    ax.set_xlabel(
        r"$x$ [$\mu$m]"
    )

    ax.set_ylabel(
        r"$z$ [$\mu$m]"
    )

    ax.set_title(
        r"Electric-field lines; opacity $\propto |\mathbf{E}|$"
    )

    ax.set_aspect("auto")
    ax.grid(False)

    if xlim_um is not None:
        ax.set_xlim(*xlim_um)

    if zlim_um is not None:
        ax.set_ylim(*zlim_um)

    else:
        ax.set_ylim(
            z_um.min(),
            z_um.max()
        )

    # Same range as your previous figure
    ax.set_ylim(-0.6, 0.6)


    # Store normalization for debugging/reproducibility
    ax._field_visibility_limits_MVm = (
        E_low,
        E_high
    )

    fig.tight_layout()

    return fig, ax

fig_field, ax_field = plot_xz_field_lines(field_xz, p)
plt.savefig("field_lines.pdf")
plt.show()

In [ ]:
# ============================================================
# Gauss-law benchmark for the x-z electric field
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# Coordinates and fields in SI units
# ------------------------------------------------------------

x_nm = np.asarray(field_xz["x_nm"])
z_nm = np.asarray(field_xz["z_nm"])

x_m = x_nm * 1e-9
z_m = z_nm * 1e-9

# MV/m -> V/m
Ex = np.asarray(field_xz["Ex_MVm"]) * 1e6
Ez = np.asarray(field_xz["Ez_MVm"]) * 1e6


# ------------------------------------------------------------
# Numerical divergence
# ------------------------------------------------------------

dEx_dx = np.gradient(
    Ex,
    x_m,
    axis=1,
    edge_order=2,
)

dEz_dz = np.gradient(
    Ez,
    z_m,
    axis=0,
    edge_order=2,
)

divE = dEx_dx + dEz_dz


# ------------------------------------------------------------
# Relative divergence
#
# This measures how well the two derivatives cancel.
#
# 0   -> perfect cancellation
# 1   -> no cancellation
# ------------------------------------------------------------

denom = (
    np.abs(dEx_dx)
    + np.abs(dEz_dz)
    + 1e-30
)

rel_div = np.abs(divE) / denom


# ------------------------------------------------------------
# Construct charge-free mask
# ------------------------------------------------------------

X_nm, Z_nm = np.meshgrid(x_nm, z_nm)

mask = np.ones_like(divE, dtype=bool)

# Distance to exclude around physical charge sheets
exclude_z_nm = 10.0
exclude_x_nm = 10.0

# 2DEG at z = 0
mask &= np.abs(Z_nm) > exclude_z_nm

# Donor sheets at +/- dist0_d
z_d = 150

mask &= np.abs(Z_nm - z_d) > exclude_z_nm
mask &= np.abs(Z_nm + z_d) > exclude_z_nm


# Surface charges near x = +/- d_S at z = 0.
# Since z=0 has already been excluded, this is mostly optional,
# but it is useful if the surface charge has a finite vertical width.
dS = float(p.d_S)

surface_region = (
    (np.abs(X_nm - dS) < exclude_x_nm)
    | (np.abs(X_nm + dS) < exclude_x_nm)
)

surface_region &= np.abs(Z_nm) < 2*exclude_z_nm

mask &= ~surface_region


# ------------------------------------------------------------
# Exclude numerical boundaries
# ------------------------------------------------------------

n_edge_x = 10
n_edge_z = 10

mask[:, :n_edge_x] = False
mask[:, -n_edge_x:] = False
mask[:n_edge_z, :] = False
mask[-n_edge_z:, :] = False


# ------------------------------------------------------------
# If using metallic plates at +/- dist, avoid the immediate
# neighbourhood of the boundaries as well
# ------------------------------------------------------------

if res["dist"] is not None:

    gate_dist = float(res["dist"])

    gate_margin_nm = 5.0

    mask &= np.abs(Z_nm - gate_dist) > gate_margin_nm
    mask &= np.abs(Z_nm + gate_dist) > gate_margin_nm


# ------------------------------------------------------------
# Statistics in charge-free region
# ------------------------------------------------------------

print("GAUSS-LAW BENCHMARK")
print("-------------------")

print(
    "median |dEx/dx| :",
    f"{np.nanmedian(np.abs(dEx_dx[mask])):.4e}",
    "V/m^2"
)

print(
    "median |dEz/dz| :",
    f"{np.nanmedian(np.abs(dEz_dz[mask])):.4e}",
    "V/m^2"
)

print(
    "median |div E|  :",
    f"{np.nanmedian(np.abs(divE[mask])):.4e}",
    "V/m^2"
)

print()

print(
    "median relative divergence :",
    f"{np.nanmedian(rel_div[mask]):.4e}"
)

print(
    "95% relative divergence    :",
    f"{np.nanpercentile(rel_div[mask], 95):.4e}"
)

print(
    "99% relative divergence    :",
    f"{np.nanpercentile(rel_div[mask], 99):.4e}"
)


# ------------------------------------------------------------
# Plot relative divergence
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(13, 5))

plot_rel = np.where(mask, rel_div, np.nan)

pcm = ax.pcolormesh(
    x_nm * 1e-3,
    z_nm * 1e-3,
    plot_rel,
    shading="auto",
    cmap="magma",
    vmin=0.0,
    vmax=0.1,
)

cbar = fig.colorbar(pcm, ax=ax)

cbar.set_label(
    r"$|\nabla\cdot\mathbf{E}|/"
    r"(|\partial_x E_x|+|\partial_z E_z|)$"
)

ax.set_xlabel(r"$x$ [$\mu$m]")
ax.set_ylabel(r"$z$ [$\mu$m]")

ax.set_title(
    "Gauss-law residual in charge-free regions"
)

plt.tight_layout()
plt.show()